In [12]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

In [2]:
ROOT = Path.cwd().parent

RUNTIME_DIR = ROOT / "dataset" / "runtime"
CHECKPOINT_DIR = ROOT / "checkpoints"

INPUT_PATH = RUNTIME_DIR / "comments_aspects.csv"
OUTPUT_PATH = RUNTIME_DIR / "tfidf_predictions.csv"

VECTORIZER_PATH = CHECKPOINT_DIR / "tfidf_vectorizer.joblib"
CLASSIFIER_PATH = CHECKPOINT_DIR / "tfidf_classifier.joblib"
MODEL_INFO_PATH = CHECKPOINT_DIR / "tfidf_model_info.joblib"

print("Input aspect CSV:", INPUT_PATH)
print("Vectorizer:", VECTORIZER_PATH)
print("Classifier:", CLASSIFIER_PATH)
print("Output predictions:", OUTPUT_PATH)

Input aspect CSV: c:\New folder\Projects\sentiment analysis\dataset\runtime\comments_aspects.csv
Vectorizer: c:\New folder\Projects\sentiment analysis\checkpoints\tfidf_vectorizer.joblib
Classifier: c:\New folder\Projects\sentiment analysis\checkpoints\tfidf_classifier.joblib
Output predictions: c:\New folder\Projects\sentiment analysis\dataset\runtime\tfidf_predictions.csv


In [3]:
required_paths = [
    INPUT_PATH,
    VECTORIZER_PATH,
    CLASSIFIER_PATH
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required file(s) missing:\n"
        + "\n".join(str(path) for path in missing_paths)
        + "\n\nRun these notebooks first:\n"
        "1. 01_train_tfidf.ipynb\n"
        "2. 03_extract_aspects.ipynb"
    )

print("All required files found.")

All required files found.


In [4]:
vectorizer = joblib.load(VECTORIZER_PATH)
classifier = joblib.load(CLASSIFIER_PATH)

print("Vectorizer loaded:", type(vectorizer))
print("Classifier loaded:", type(classifier))

print("\nClassifier classes:")
print(classifier.classes_)

Vectorizer loaded: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
Classifier loaded: <class 'sklearn.svm._classes.LinearSVC'>

Classifier classes:
['conflict' 'negative' 'neutral' 'positive']


In [ ]:
if MODEL_INFO_PATH.exists():
    model_info = joblib.load(MODEL_INFO_PATH)

    print("\nSaved model information:")
    print(model_info)
else:
    model_info = {
        "model_name": "unknown",
        "input_format": "aspect_term [SEP] text"
    }

    print(
        "\ntfidf_model_info.joblib was not found. "
        "Continuing with model inference."
    )


Saved model information:
{'model_name': 'linear_svc', 'classes': ['conflict', 'negative', 'neutral', 'positive'], 'accuracy': 0.6972477064220184, 'macro_f1': 0.5059936666448366, 'input_format': 'aspect_term [SEP] text'}


In [6]:
aspect_data = pd.read_csv(INPUT_PATH)

print("Original aspect-row count:", len(aspect_data))
print("Columns:", aspect_data.columns.tolist())

required_columns = {
    "comment_id",
    "text",
    "aspect_term"
}

missing_columns = required_columns - set(aspect_data.columns)

if missing_columns:
    raise ValueError(
        "comments_aspects.csv is missing columns: "
        + ", ".join(sorted(missing_columns))
    )

aspect_data = aspect_data.dropna(
    subset=["comment_id", "text", "aspect_term"]
).copy()

aspect_data["text"] = (
    aspect_data["text"]
    .astype(str)
    .str.strip()
)

aspect_data["aspect_term"] = (
    aspect_data["aspect_term"]
    .astype(str)
    .str.strip()
    .str.lower()
)

aspect_data = aspect_data[
    aspect_data["text"].ne("") &
    aspect_data["aspect_term"].ne("")
].copy()

aspect_data = aspect_data.reset_index(drop=True)

print("Usable aspect-row count:", len(aspect_data))

display(aspect_data.head(10))

Original aspect-row count: 11
Columns: ['comment_id', 'text', 'raw_aspect', 'aspect_term']
Usable aspect-row count: 11


,comment_id,text,raw_aspect,aspect_term
0,c001,The laptop is excellent for school work and br...,8gb ram,8gb ram
1,c001,The laptop is excellent for school work and br...,school,school
2,c001,The laptop is excellent for school work and br...,browsing,productivity
3,c002,"Light games and older titles run fine, but AAA...",aaa,gaming performance
4,c003,"The touchscreen is useful, and Microsoft Offic...",office,office work
5,c003,"The touchscreen is useful, and Microsoft Offic...",microsoft office,productivity
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,touchscreen
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,gaming performance
8,c004,"The laptop has poor gaming performance, althou...",productivity,productivity
9,c005,"The battery life is decent, but storage fills ...",storage,storage


In [7]:
aspect_data["model_input"] = (
    aspect_data["aspect_term"]
    + " [SEP] "
    + aspect_data["text"]
)

display(
    aspect_data[
        [
            "comment_id",
            "aspect_term",
            "model_input"
        ]
    ].head(10)
)

,comment_id,aspect_term,model_input
0,c001,8gb ram,8gb ram [SEP] The laptop is excellent for scho...
1,c001,school,school [SEP] The laptop is excellent for schoo...
2,c001,productivity,productivity [SEP] The laptop is excellent for...
3,c002,gaming performance,gaming performance [SEP] Light games and older...
4,c003,office work,"office work [SEP] The touchscreen is useful, a..."
5,c003,productivity,"productivity [SEP] The touchscreen is useful, ..."
6,c003,touchscreen,"touchscreen [SEP] The touchscreen is useful, a..."
7,c004,gaming performance,gaming performance [SEP] The laptop has poor g...
8,c004,productivity,productivity [SEP] The laptop has poor gaming ...
9,c005,storage,"storage [SEP] The battery life is decent, but ..."


In [8]:
X_inference = vectorizer.transform(
    aspect_data["model_input"]
)

print("Inference matrix type:", type(X_inference))
print("Inference matrix shape:", X_inference.shape)

Inference matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Inference matrix shape: (11, 5000)


In [9]:
aspect_data["predicted_polarity"] = classifier.predict(
    X_inference
)

In [10]:
def get_prediction_confidence(classifier, X):
    """
    Return confidence scores in the range 0–1 when possible.

    Logistic Regression:
        Maximum predicted class probability.

    LinearSVC:
        Softmax-normalized decision scores.
        This is useful for ranking but is not a calibrated probability.
    """
    if hasattr(classifier, "predict_proba"):
        probabilities = classifier.predict_proba(X)

        confidence_scores = probabilities.max(axis=1)

        all_probabilities = [
            {
                class_name: round(float(probability), 4)
                for class_name, probability in zip(
                    classifier.classes_,
                    row
                )
            }
            for row in probabilities
        ]

        confidence_type = "probability"

    elif hasattr(classifier, "decision_function"):
        decision_scores = classifier.decision_function(X)

        # Binary LinearSVC has one decision score per row.
        if decision_scores.ndim == 1:
            decision_scores = np.column_stack(
                [-decision_scores, decision_scores]
            )

        shifted_scores = (
            decision_scores
            - decision_scores.max(
                axis=1,
                keepdims=True
            )
        )

        exp_scores = np.exp(shifted_scores)

        softmax_scores = exp_scores / exp_scores.sum(
            axis=1,
            keepdims=True
        )

        confidence_scores = softmax_scores.max(axis=1)

        all_probabilities = [
            {
                class_name: round(float(probability), 4)
                for class_name, probability in zip(
                    classifier.classes_,
                    row
                )
            }
            for row in softmax_scores
        ]

        confidence_type = "decision-score softmax"

    else:
        confidence_scores = np.full(
            shape=X.shape[0],
            fill_value=np.nan
        )

        all_probabilities = [
            {}
            for _ in range(X.shape[0])
        ]

        confidence_type = "unavailable"

    return confidence_scores, all_probabilities, confidence_type

In [13]:
confidence, all_probabilities, confidence_type = (
    get_prediction_confidence(
        classifier,
        X_inference
    )
)

aspect_data["confidence"] = confidence
aspect_data["all_class_scores"] = all_probabilities

print("Confidence method:", confidence_type)

Confidence method: decision-score softmax


In [14]:
prediction_columns = [
    "comment_id",
    "text",
    "raw_aspect",
    "aspect_term",
    "predicted_polarity",
    "confidence",
    "all_class_scores"
]

# raw_aspect may not exist if you created comments_aspects.csv manually.
prediction_columns = [
    column
    for column in prediction_columns
    if column in aspect_data.columns
]

tfidf_predictions = aspect_data[
    prediction_columns
].copy()

tfidf_predictions["model"] = (
    model_info.get(
        "model_name",
        "tfidf_classifier"
    )
)

tfidf_predictions = tfidf_predictions[
    [
        "comment_id",
        "text",
        *(
            ["raw_aspect"]
            if "raw_aspect" in tfidf_predictions.columns
            else []
        ),
        "aspect_term",
        "predicted_polarity",
        "confidence",
        "all_class_scores",
        "model"
    ]
]

tfidf_predictions = tfidf_predictions.sort_values(
    by=["comment_id", "aspect_term"]
).reset_index(drop=True)

display(tfidf_predictions.head(20))

,comment_id,text,raw_aspect,aspect_term,predicted_polarity,confidence,all_class_scores,model
0,c001,The laptop is excellent for school work and br...,8gb ram,8gb ram,negative,0.536259,"{'conflict': 0.0925, 'negative': 0.5363, 'neut...",linear_svc
1,c001,The laptop is excellent for school work and br...,browsing,productivity,positive,0.638705,"{'conflict': 0.0883, 'negative': 0.1402, 'neut...",linear_svc
2,c001,The laptop is excellent for school work and br...,school,school,positive,0.480804,"{'conflict': 0.096, 'negative': 0.2914, 'neutr...",linear_svc
3,c002,"Light games and older titles run fine, but AAA...",aaa,gaming performance,negative,0.510392,"{'conflict': 0.0997, 'negative': 0.5104, 'neut...",linear_svc
4,c003,"The touchscreen is useful, and Microsoft Offic...",office,office work,positive,0.532008,"{'conflict': 0.0967, 'negative': 0.1684, 'neut...",linear_svc
5,c003,"The touchscreen is useful, and Microsoft Offic...",microsoft office,productivity,positive,0.566816,"{'conflict': 0.0925, 'negative': 0.0871, 'neut...",linear_svc
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,touchscreen,positive,0.421551,"{'conflict': 0.1152, 'negative': 0.1987, 'neut...",linear_svc
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,gaming performance,negative,0.711015,"{'conflict': 0.0783, 'negative': 0.711, 'neutr...",linear_svc
8,c004,"The laptop has poor gaming performance, althou...",productivity,productivity,negative,0.357693,"{'conflict': 0.1167, 'negative': 0.3577, 'neut...",linear_svc
9,c005,"The battery life is decent, but storage fills ...",battery,battery,negative,0.527033,"{'conflict': 0.1329, 'negative': 0.527, 'neutr...",linear_svc


In [15]:
tfidf_predictions.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved TF-IDF predictions to:")
print(OUTPUT_PATH)

print("\nTotal predicted aspect-polarity pairs:")
print(len(tfidf_predictions))

Saved TF-IDF predictions to:
c:\New folder\Projects\sentiment analysis\dataset\runtime\tfidf_predictions.csv

Total predicted aspect-polarity pairs:
11


In [16]:
aspect_summary = (
    tfidf_predictions
    .groupby(
        ["aspect_term", "predicted_polarity"]
    )
    .size()
    .reset_index(name="mentions")
    .sort_values(
        ["aspect_term", "mentions"],
        ascending=[True, False]
    )
)

display(aspect_summary)

,aspect_term,predicted_polarity,mentions
0,8gb ram,negative,1
1,battery,negative,1
2,gaming performance,negative,2
3,office work,positive,1
5,productivity,positive,2
4,productivity,negative,1
6,school,positive,1
7,storage,negative,1
8,touchscreen,positive,1


In [17]:
low_confidence = (
    tfidf_predictions
    .sort_values("confidence", ascending=True)
    .head(20)
)

display(
    low_confidence[
        [
            "comment_id",
            "text",
            "aspect_term",
            "predicted_polarity",
            "confidence"
        ]
    ]
)

,comment_id,text,aspect_term,predicted_polarity,confidence
8,c004,"The laptop has poor gaming performance, althou...",productivity,negative,0.357693
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,positive,0.421551
10,c005,"The battery life is decent, but storage fills ...",storage,negative,0.454091
2,c001,The laptop is excellent for school work and br...,school,positive,0.480804
3,c002,"Light games and older titles run fine, but AAA...",gaming performance,negative,0.510392
9,c005,"The battery life is decent, but storage fills ...",battery,negative,0.527033
4,c003,"The touchscreen is useful, and Microsoft Offic...",office work,positive,0.532008
0,c001,The laptop is excellent for school work and br...,8gb ram,negative,0.536259
5,c003,"The touchscreen is useful, and Microsoft Offic...",productivity,positive,0.566816
1,c001,The laptop is excellent for school work and br...,productivity,positive,0.638705
